In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_18.pickle

In [ ]:
%%RecordEvent
%%time
### cell 19 ###

### cell 19 (optimized for cudf)

def bar_chart_multiple_choice_28(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    # response_counts is already a cudf.Series
    response_counts.index.name = ''
    # turn into a one‐column GPU DataFrame and write directly to CSV on GPU
    df_counts = response_counts.to_frame(name='count')
    df_counts.to_csv(
        f'/home/dias-benchmarks/notebooks/paultimothymooney'
        f'/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/{title}.csv',
        index=True
    )
    # compute max for chart scaling all on GPU
    tmp_cmp = (response_counts.iloc[:num_choices] * 1.2).max()


def count_then_return_percent_28(dataframe, x_axis_title):
    # all operations use cudf under the hood
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    # drop NaNs to get non-null total
    total_nonnull = counts.dropna().sum()
    # compute percentage and round on GPU
    percentages = (counts / total_nonnull * 100).round(1)
    return percentages

# single mapping replacement on GPU
replacement_map = {
    "Bachelorâ\x80\x99s degree": "Bachelor's degree",
    "Masterâ\x80\x99s degree": "Master's degree",
    "Some college/university study without earning a bachelorâ\x80\x99s degree":
        "Some university study but without earning a degree"
}
# do not modify responses_df_2022 in-place
cleaned_series = responses_df_2022[question_of_interest].replace(replacement_map)

# compute percentages entirely on GPU and select the desired order
percentages = (
    count_then_return_percent_28(
        # wrap series in a one‐column cudf.DataFrame
        responses_df_2022[[question_of_interest]].assign(**{question_of_interest: cleaned_series}),
        question_of_interest
    )
    .sort_index()
    .reindex(responses_in_order)
)

title_for_chart = 'Most common degree types on Kaggle'
title_for_y_axis = '% of respondents'
bar_chart_multiple_choice_28(
    response_counts=percentages,
    title=title_for_chart,
    y_axis_title=title_for_y_axis,
    orientation=orientation_for_chart
)

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_19_try_9.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_19_try_9.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[19], f)


In [ ]:
opt_output = Out.get(4)